In [1]:
'''
EDA
basic nlp topic analysis with NMF and LDA

Lot to do here. not nearly enough exposure for parameter searching
also gensim not working on python 3.14 so need an earlier version
would want to do some sweeping through K for the various methods and testing different random seeds and what not

# ---------------------------------------------------------------------------
# Tips
# ---------------------------------------------------------------------------
# Custom stopwords on top of sklearn's English defaults:
#   from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
#   sw = set(ENGLISH_STOP_WORDS) | {'vehicle', 'av', 'driver', 'incident'}
#   lda_sklearn(narratives, stop_words=list(sw))
#
# N-grams (sklearn only): lda_sklearn(narratives, ngram_range=(1, 2))
#
# Tiny corpus / vocab fully filtered: relax min_df/no_below (1) and
# max_df/no_above (1.0). The pipelines raise ValueError with an
# actionable message rather than crashing inside the library.
#
# Reproducibility: all four accept `random_state` (default 0).


'''

"\nEDA\nbasic nlp topic analysis with NMF and LDA\n\n# ---------------------------------------------------------------------------\n# Tips\n# ---------------------------------------------------------------------------\n# Custom stopwords on top of sklearn's English defaults:\n#   from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\n#   sw = set(ENGLISH_STOP_WORDS) | {'vehicle', 'av', 'driver', 'incident'}\n#   lda_sklearn(narratives, stop_words=list(sw))\n#\n# N-grams (sklearn only): lda_sklearn(narratives, ngram_range=(1, 2))\n#\n# Tiny corpus / vocab fully filtered: relax min_df/no_below (1) and\n# max_df/no_above (1.0). The pipelines raise ValueError with an\n# actionable message rather than crashing inside the library.\n#\n# Reproducibility: all four accept `random_state` (default 0).\n\n\n"

In [24]:
import pandas as pd
import numpy as np

import sys
sys.path.append('..')

import eda_utils_sgo
import eda_utils_nlp
import eda_utils_dedupe
import eda_utils_treatment
from eda_utils_topics import lda_sklearn, nmf_sklearn, lda_gensim, nmf_gensim


pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 300)

In [3]:
%load_ext autoreload
%autoreload 2

## Load Data and Treat Data

In [4]:
from pathlib import Path
repo_root = Path.cwd().parents[1]  # eda/ADS_to_2026_03_16 -> repo root
data_dir = repo_root / 'data' / 'nhtsa'
paths = [
    data_dir / 'SGO-2021-01_Incident_Reports_ADS_to_2025_06_16.csv',
    data_dir / 'SGO-2021-01_Incident_Reports_ADS_2025_06_16_to_2026_03_16.csv',
]

In [5]:
ads_df = eda_utils_sgo.load_and_concat_csvs(paths)

Only in SGO-2021-01_Incident_Reports_ADS_to_2025_06_16.csv:
  ADAS/ADS Hardware Version
  ADAS/ADS Hardware Version - Unk
  ADAS/ADS Hardware Version CBI
  ADAS/ADS Software Version
  ADAS/ADS Software Version - Unk
  ADAS/ADS Software Version CBI
  ADAS/ADS System Version
  ADAS/ADS System Version - Unk
  ADAS/ADS System Version CBI
  ADS Equipped?
  CP Any Air Bags Deployed?
  CP Was Vehicle Towed?
  Federal Reg. Exemption - No
  Federal Reg. Exemption - Unk
  Federal Regulatory Exemption?
  Inv. Officer Email - Unknown
  Inv. Officer Name - Unknown
  Inv. Officer Phone - Unknown
  Investigating Officer Email
  Investigating Officer Name
  Investigating Officer Phone
  Law Enforcement Investigating?
  Lighting
  Mileage
  Mileage - Unknown
  Notice Received Date
  Other Federal Reg. Exemption
  Other Reporting Entities?
  Other Reporting Entities? - NA
  Other Reporting Entities? - Unk
  Posted Speed Limit (MPH)
  Posted Speed Limit - Unknown
  Property Damage?
  Rep Ent Or Mfr Inves

In [ ]:
entity_column = 'master_entity'

In [26]:
# add overall entity and drop duplicates

treated_df = eda_utils_dedupe.dedupe_same_incident(ads_df.copy(), verbose=True)

# to do: more treatment of various columns

treated_df = eda_utils_treatment.apply_all_treatments(treated_df)

new_cols = [c for c in treated_df.columns if c not in ads_df.columns]
print(new_cols)
treated_df[['Operating Entity', 'Operating Entity Clean',
        'Reporting Entity', 'Reporting Entity Clean',
        'master_entity',
        'Make', 'Model', 'Make Model',
        'State', 'State Clean']].head(10)




dedupe_same_incident: 3120 -> 2344 rows (776 duplicates collapsed)
['Narrative - Same Incident ID', 'Operating Entity Clean', 'Reporting Entity Clean', 'Investigating Agency Clean', 'State or Local Permit Clean', 'Make Clean', 'Model Clean', 'State Clean', 'master_entity', 'Make Model']


## Topic Analysis

In [7]:
narratives = treated_df['Narrative'].copy()

### LDA sklearn

In [27]:
# ---------------------------------------------------------------------------
# 1) CountVectorizer -> sklearn LatentDirichletAllocation
# ---------------------------------------------------------------------------
topics_df, doc_topic, doc_index = lda_sklearn(
    narratives,
    n_topics=8,
    top_n_words=12,
    min_df=5,
    max_df=0.9,
    # learning_method='batch' is the default; switch to 'online' for huge corpora.
)
print('lda_sklearn topics_df:', topics_df.shape, 'doc_topic:', doc_topic.shape)

with pd.option_context('display.max_colwidth', None):
    display(topics_df)

# Assign dominant topic back to ads_df by doc_index.
dominant = pd.Series(doc_topic.argmax(axis=1), index=doc_index,
                     name='lda_sklearn_topic')
treated_df_topics = treated_df.join(dominant)
print(treated_df_topics['lda_sklearn_topic'].value_counts(dropna=False).head())

lda_sklearn topics_df: (8, 3) doc_topic: (2342, 8)


,topic,top_words,top_weights
0,0,"aurora, truck, incident, lane, driver, shuttle, collision, autonomy, approximately, damage, operator, vehicles","[0.023630447167118512, 0.020710112108549783, 0.01987428894240935, 0.01798806036667145, 0.017948744655518677, 0.016935570134731615, 0.015904960570943344, 0.015579026514110353, 0.012789755960561205, 0.011257691998083798, 0.011067036129770134, 0.010348291376037306]"
1,1,"lane, right, motorcycle, safety, driver, autonomous, left, curb, turn, tire, motional, stop","[0.030414802951670903, 0.021237117590314946, 0.02096252053815342, 0.017668103135296636, 0.01600492870290722, 0.015674933815825746, 0.014004083258479839, 0.013178327905505336, 0.012834063457240684, 0.012102756425653113, 0.012015974673327023, 0.011762245354350996]"
2,2,"av, report, information, business, confidential, redacted, contain, submitted, unchanged, initial, cyclist, content","[0.06451152772096777, 0.05229701584904761, 0.042963309721660375, 0.0276383038902079, 0.02722190976499781, 0.026681385218620703, 0.02625031830590493, 0.021121238011027353, 0.018061180201317608, 0.018015645941493956, 0.017316772883309267, 0.015562795231992619]"
3,3,"waymo, av, reporting, passenger, autonomous, car, time, contact, collision, standing, level, traveling","[0.12227438745070715, 0.07691945454195033, 0.025351010325236505, 0.0245473766860795, 0.024130682890131448, 0.016349717000178354, 0.013823678727070035, 0.01323220175907023, 0.012934328667710061, 0.012927679573963187, 0.012889432603231164, 0.012883367921750297]"
4,4,"gm, cruise, report, incident, submission, order, llc, 2021, based, reference, fields, general","[0.0866811845952393, 0.08500633019326337, 0.052958300974864536, 0.03941008908664323, 0.030554513937385574, 0.029573629462762582, 0.022906057485786595, 0.022460166173469246, 0.019616269039705894, 0.019543353963346614, 0.019399886220245546, 0.01937674628833518]"
5,5,"truck, av, pickup, driver, test, autonomous, mode, transdev, lane, avride, report, left","[0.08063460654059279, 0.07701739693649715, 0.061707948956197856, 0.044340970082524876, 0.028986342428399654, 0.02443551479702795, 0.02426544108402091, 0.020154746560919312, 0.019259668261212264, 0.017581321176251923, 0.016452898508056365, 0.015705303396579266]"
6,6,"gm, cruise, report, incident, submission, order, based, general, reference, facts, second, 2023","[0.09032461563136783, 0.06876051014818522, 0.05395470557225167, 0.04347476273500875, 0.03211154056753775, 0.031947333799442416, 0.021471871220366128, 0.020911973252893075, 0.020904416893629076, 0.020856757052538998, 0.020830351099545417, 0.0208066341563404]"
7,7,"zoox, right, car, left, lane, rear, damage, traffic, passenger, turn, injuries, stopped","[0.059479153856143795, 0.024661260561910786, 0.021185965728146516, 0.020989393221185836, 0.02017847940860534, 0.020169230257761617, 0.01995341541006253, 0.01787909102400332, 0.017374707709092232, 0.017310224430830384, 0.01591750889775281, 0.015782391412954965]"


lda_sklearn_topic
3.0    1784
7.0     170
6.0     111
1.0      70
0.0      66
Name: count, dtype: int64


In [28]:
treated_df_topics.groupby([entity_column, 'lda_sklearn_topic', ])['Report ID'].agg('count').reset_index()

,master_entity,lda_sklearn_topic,Report ID
0,Admt Vwgoa,1.0,1
1,Admt Vwgoa,7.0,9
2,Apollo Autonomous Driving USA,1.0,1
3,Apollo Autonomous Driving USA,7.0,2
4,Apple,1.0,3
5,Argo AI,2.0,20
6,Aurora Operations,0.0,21
7,Autox Technologies,3.0,1
8,Avride,1.0,2
9,Avride,5.0,30


In [12]:
with pd.option_context('display.max_colwidth', None):
    display(topics_df)

,topic,top_words,top_weights
0,0,"aurora, truck, incident, lane, driver, shuttle, collision, autonomy, approximately, damage, operator, vehicles","[0.023630447167118512, 0.020710112108549783, 0.01987428894240935, 0.01798806036667145, 0.017948744655518677, 0.016935570134731615, 0.015904960570943344, 0.015579026514110353, 0.012789755960561205, 0.011257691998083798, 0.011067036129770134, 0.010348291376037306]"
1,1,"lane, right, motorcycle, safety, driver, autonomous, left, curb, turn, tire, motional, stop","[0.030414802951670903, 0.021237117590314946, 0.02096252053815342, 0.017668103135296636, 0.01600492870290722, 0.015674933815825746, 0.014004083258479839, 0.013178327905505336, 0.012834063457240684, 0.012102756425653113, 0.012015974673327023, 0.011762245354350996]"
2,2,"av, report, information, business, confidential, redacted, contain, submitted, unchanged, initial, cyclist, content","[0.06451152772096777, 0.05229701584904761, 0.042963309721660375, 0.0276383038902079, 0.02722190976499781, 0.026681385218620703, 0.02625031830590493, 0.021121238011027353, 0.018061180201317608, 0.018015645941493956, 0.017316772883309267, 0.015562795231992619]"
3,3,"waymo, av, reporting, passenger, autonomous, car, time, contact, collision, standing, level, traveling","[0.12227438745070715, 0.07691945454195033, 0.025351010325236505, 0.0245473766860795, 0.024130682890131448, 0.016349717000178354, 0.013823678727070035, 0.01323220175907023, 0.012934328667710061, 0.012927679573963187, 0.012889432603231164, 0.012883367921750297]"
4,4,"gm, cruise, report, incident, submission, order, llc, 2021, based, reference, fields, general","[0.0866811845952393, 0.08500633019326337, 0.052958300974864536, 0.03941008908664323, 0.030554513937385574, 0.029573629462762582, 0.022906057485786595, 0.022460166173469246, 0.019616269039705894, 0.019543353963346614, 0.019399886220245546, 0.01937674628833518]"
5,5,"truck, av, pickup, driver, test, autonomous, mode, transdev, lane, avride, report, left","[0.08063460654059279, 0.07701739693649715, 0.061707948956197856, 0.044340970082524876, 0.028986342428399654, 0.02443551479702795, 0.02426544108402091, 0.020154746560919312, 0.019259668261212264, 0.017581321176251923, 0.016452898508056365, 0.015705303396579266]"
6,6,"gm, cruise, report, incident, submission, order, based, general, reference, facts, second, 2023","[0.09032461563136783, 0.06876051014818522, 0.05395470557225167, 0.04347476273500875, 0.03211154056753775, 0.031947333799442416, 0.021471871220366128, 0.020911973252893075, 0.020904416893629076, 0.020856757052538998, 0.020830351099545417, 0.0208066341563404]"
7,7,"zoox, right, car, left, lane, rear, damage, traffic, passenger, turn, injuries, stopped","[0.059479153856143795, 0.024661260561910786, 0.021185965728146516, 0.020989393221185836, 0.02017847940860534, 0.020169230257761617, 0.01995341541006253, 0.01787909102400332, 0.017374707709092232, 0.017310224430830384, 0.01591750889775281, 0.015782391412954965]"


In [29]:
# ---------------------------------------------------------------------------
# 1) CountVectorizer -> sklearn LatentDirichletAllocation
# ---------------------------------------------------------------------------
topics_df, doc_topic, doc_index = lda_sklearn(
    narratives,
    n_topics=6,
    top_n_words=12,
    min_df=10,
    max_df=0.8,
    # learning_method='batch' is the default; switch to 'online' for huge corpora.
    random_state=1 
)
print('lda_sklearn topics_df:', topics_df.shape, 'doc_topic:', doc_topic.shape)

with pd.option_context('display.max_colwidth', None):
    display(topics_df)

# Assign dominant topic back to ads_df by doc_index.
dominant = pd.Series(doc_topic.argmax(axis=1), index=doc_index,
                     name='lda_sklearn_topic')
treated_df_topics = treated_df.join(dominant)
print(treated_df_topics['lda_sklearn_topic'].value_counts(dropna=False).head())

lda_sklearn topics_df: (6, 3) doc_topic: (2342, 6)


,topic,top_words,top_weights
0,0,"waymo, driver, report, test, passenger, transdev, crash, correct, collision, impact, engaged, additional","[0.11927126864506003, 0.03439421123061574, 0.03042167608062653, 0.0201578173442337, 0.017474927757353785, 0.014265258581027875, 0.014071087185080303, 0.01401151961163728, 0.013621305813779652, 0.013618765787449817, 0.013400985269319456, 0.013378713547131329]"
1,1,"gm, cruise, report, incident, submission, based, llc, reference, facts, fields, responses, second","[0.10263196705391364, 0.08479188804296292, 0.06316468699154297, 0.04862493112902727, 0.036391035021939494, 0.023843204400452032, 0.02382524089216089, 0.023670327968410412, 0.02343763280657778, 0.023360052668329433, 0.022817158547647945, 0.01936455680127877]"
2,2,"waymo, truck, lane, left, right, pickup, heavy, turn, traveling, stopped, began, collision","[0.09033329879213071, 0.0826705863208406, 0.052222488504353924, 0.045251975724187335, 0.035315889966741135, 0.03426641817938854, 0.029384026925288412, 0.025694436828902413, 0.0199188982974147, 0.01455313537961122, 0.012771122569259427, 0.012684435583468035]"
3,3,"waymo, suv, rear, collision, impact, level, request, involving, crash, supplement, available, correct","[0.1935323884073894, 0.05817735903352613, 0.023753248860776153, 0.021981786996672508, 0.021012744984467188, 0.020953169469210213, 0.020728099947245934, 0.020715968332510267, 0.020574243155496698, 0.020380598029853886, 0.020339688907193374, 0.020329459824252622]"
4,4,"waymo, passenger, car, traveling, lane, collision, level, engaged, crash, request, impact, additional","[0.16199369273238315, 0.058590675727344554, 0.04846598105287308, 0.019920422471152256, 0.01917475356446408, 0.017628056677124915, 0.01734424275675934, 0.0173366021009665, 0.017335894680583983, 0.01727143071738454, 0.0172702353256026, 0.0171966550431199]"
5,5,"zoox, lane, right, driver, rear, injuries, left, reported, operator, stop, traveling, turn","[0.03550583151393327, 0.02240914696409102, 0.020495052022486866, 0.018703176729941654, 0.016251370734157188, 0.015283407267680545, 0.014886827045841461, 0.014849976479630553, 0.014355529672442526, 0.014316049146102357, 0.013725618999374722, 0.01228476745090269]"


lda_sklearn_topic
4.0    893
3.0    485
5.0    359
0.0    248
2.0    208
Name: count, dtype: int64


In [30]:
treated_df_topics.groupby([entity_column, 'lda_sklearn_topic', ])['Report ID'].agg('count').reset_index()

,master_entity,lda_sklearn_topic,Report ID
0,Admt Vwgoa,5.0,10
1,Apollo Autonomous Driving USA,5.0,3
2,Apple,5.0,3
3,Argo AI,0.0,1
4,Argo AI,5.0,19
5,Aurora Operations,5.0,21
6,Autox Technologies,5.0,1
7,Avride,2.0,29
8,Avride,5.0,10
9,Beep,5.0,14


### NMF sklearn
* ug should refactor this to show more possible options

In [18]:
# ---------------------------------------------------------------------------
# 2) TfidfVectorizer -> sklearn NMF
# ---------------------------------------------------------------------------
topics_df, doc_topic, doc_index = nmf_sklearn(
    narratives,
    n_topics=8,
    top_n_words=12,
    min_df=5,
    max_df=0.9,
)
print('nmf_sklearn topics_df:', topics_df.shape)
with pd.option_context('display.max_colwidth', None):
    display(topics_df)

# Full per-doc topic-weight DataFrame indexed back to the original rows.
doc_topic_df = pd.DataFrame(
    doc_topic,
    index=doc_index,
    columns=[f'nmf_topic_{k}' for k in range(doc_topic.shape[1])],
)
display(doc_topic_df.head())

nmf_sklearn topics_df: (8, 3)


,topic,top_words,top_weights
0,0,"waymo, suv, av, reporting, autonomous, lane, left, stopped, traveling, right, rear, collision","[0.07998045464813283, 0.060004016805098294, 0.05111324231963409, 0.015684173829259514, 0.015307406199902498, 0.012491153540602195, 0.011251017700609475, 0.010799332755561863, 0.010563515611434218, 0.009120507046983037, 0.009034544311978763, 0.008858042318216006]"
1,1,"gm, cruise, report, incident, submission, responses, facts, fields, reference, llc, based, second","[0.10467115970385982, 0.0815405800738796, 0.04229245087954465, 0.04009522305888636, 0.03416093818225574, 0.023260257711968855, 0.023121370630480666, 0.02310275265933674, 0.022920929048184668, 0.022798930012380668, 0.02257026210688515, 0.015138125202147463]"
2,2,"truck, waymo, pickup, av, heavy, autonomous, reporting, left, lane, stopped, traveling, right","[0.07703206060583241, 0.0665906995936207, 0.0472207888443109, 0.04427973931516228, 0.03950065679710013, 0.012342714626377989, 0.012328260243881992, 0.011923793342260907, 0.01138138070683225, 0.010651978324215201, 0.0094099163711064, 0.009102537362449084]"
3,3,"waymo, car, passenger, av, reporting, autonomous, lane, traveling, stopped, intersection, left, rear","[0.07646084485214377, 0.05534308903477284, 0.04915986372015307, 0.04828457920888003, 0.01530846098047423, 0.014921128121094357, 0.011206423387927027, 0.01041354283206618, 0.009802377396824824, 0.009786484707296093, 0.009419708131988495, 0.008569787342330812]"
4,4,"zoox, police, called, operator, injuries, autonomy, lane, right, reported, left, turn, traffic","[0.08702025543607242, 0.01982635380262591, 0.01724247027116659, 0.016358580890251082, 0.01515739971866822, 0.014636648988280182, 0.014454923419869746, 0.014114184559447402, 0.012992685035091845, 0.012034828090325057, 0.01181081356777917, 0.01043379666561624]"
5,5,"contain, confidential, business, redacted, information, report, motional, argo, initial, submitted, content, unchanged","[0.1123939504857488, 0.11160608634354047, 0.11117038156020002, 0.09118153533034427, 0.036610934574797284, 0.013589901435173234, 0.013153193978267985, 0.011962248373624555, 0.009547085994342513, 0.009428021896599007, 0.009382764409159581, 0.00907647214377826]"
6,6,"parking, lot, waymo, av, entrance, arizona, stall, chain, mt, pavement, reporting, autonomous","[0.0889910628679612, 0.0789605731698218, 0.053815837670528804, 0.03337727734484323, 0.019193470678638563, 0.014868327729085095, 0.013891069256137315, 0.013551294978501313, 0.012857922356534062, 0.01258479425143912, 0.011263254479351099, 0.011159530321268546]"
7,7,"waymo, transdev, av, driver, test, report, seating, position, alternative, services, present, pst","[0.03544386872845417, 0.032056428435684035, 0.02877321283656836, 0.027235460677272342, 0.02607418655508616, 0.025470037417203686, 0.016669851967356333, 0.01655826255850178, 0.016091074318028704, 0.015884160593650377, 0.01586561087035233, 0.013256489010639355]"


,nmf_topic_0,nmf_topic_1,nmf_topic_2,nmf_topic_3,nmf_topic_4,nmf_topic_5,nmf_topic_6,nmf_topic_7
0,0.020650,0.006272,0.008768,0.097028,0.007710,0.071648,0.000000,0.044024
1,0.000000,0.000000,0.000000,0.000000,0.207558,0.000026,0.000000,0.000000
2,0.003539,0.000000,0.000000,0.000000,0.233995,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.041464,0.000000,0.042568,0.000733,0.000000,0.000000
4,0.000000,0.015984,0.000000,0.006549,0.046205,0.006198,0.001032,0.000000


In [22]:
topics_df, doc_topic, doc_index = nmf_sklearn(
    narratives,
    n_topics=8,
    top_n_words=15,
    min_df=20,
    max_df=0.9,
    ngram_range=(1, 3),
    random_state=2,
)
print('nmf_sklearn topics_df:', topics_df.shape)
with pd.option_context('display.max_colwidth', None):
    display(topics_df)

# Full per-doc topic-weight DataFrame indexed back to the original rows.
doc_topic_df = pd.DataFrame(
    doc_topic,
    index=doc_index,
    columns=[f'nmf_topic_{k}' for k in range(doc_topic.shape[1])],
)
display(doc_topic_df.head())
#ngram_range=(1, 1)

nmf_sklearn topics_df: (8, 3)


,topic,top_words,top_weights
0,0,"waymo, waymo av, av, suv, reporting, autonomous, collision involving suv, involving suv, suv xxx, involving suv xxx, suv contact, lane, left, xxx waymo, xxx waymo av","[0.02550475361765845, 0.016863538311312563, 0.016358531989412207, 0.016266680835977362, 0.004984146772660331, 0.004843958506506922, 0.004645004148763606, 0.004645004148763606, 0.004538834965898596, 0.004524918452650259, 0.004317622442409012, 0.004003106493465041, 0.0035302735299296128, 0.003437591242999901, 0.0034177942577592866]"
1,1,"gm, cruise, report, incident, submission, incident report, facts, cruise responses, report gm, responses, gm cruise, fields, reference, llc, based","[0.03334762591657262, 0.02592448208766825, 0.013666159370046924, 0.013049665003206009, 0.010975335885857777, 0.007430653612615332, 0.00741895266288618, 0.007410583537016129, 0.007410583537016129, 0.007410583537016129, 0.007410583537016129, 0.0073544303633110795, 0.007337806979030336, 0.007265430457592247, 0.00723974473506508]"
2,2,"waymo, passenger car, car, passenger, waymo av, av, involving passenger car, car xxx, passenger car xxx, car contact, passenger car contact, collision involving passenger, involving passenger, reporting, autonomous","[0.022580705803337934, 0.01897952538523178, 0.018961771265052527, 0.015965143763261583, 0.014764348624624985, 0.014334658000515305, 0.0054619459155534076, 0.0053614936370876805, 0.005352572792246438, 0.005331244422077697, 0.005326495131526694, 0.0052732763957133956, 0.005271699207984245, 0.004511352251960186, 0.004415907228691991]"
3,3,"zoox, zoox vehicle, police, operator, called, police called, autonomy, injuries, right, lane, reported, vehicle vehicle, injuries police, scene, injuries police called","[0.05162772844896539, 0.04779077195536093, 0.01150880694708454, 0.010784217039522861, 0.009955874318067923, 0.009640706689636559, 0.009229906162702464, 0.009145943281377885, 0.008040456379493962, 0.007773201578200667, 0.007725000258693023, 0.007570392641197294, 0.0073528966391143945, 0.006588706147142469, 0.006539413617727275]"
4,4,"contain, contain confidential, contain confidential business, redacted contain, redacted contain confidential, confidential business information, confidential, business information, confidential business, business, redacted, information, report, motional, submitted","[0.04147619652348851, 0.04147619652348851, 0.04147619652348851, 0.04147619652348851, 0.04147619652348851, 0.04133289814188781, 0.04133289814188781, 0.04133289814188781, 0.04133289814188781, 0.04114548543805824, 0.03376073021556444, 0.014044808166711728, 0.009679986835465337, 0.007554494699751175, 0.005124571082111203]"
5,5,"truck, waymo, pickup truck, pickup, waymo av, av, heavy truck, heavy, truck xxx, truck contact, involving pickup truck, collision involving pickup, involving pickup, pickup truck xxx, truck xxx near","[0.027433008670024565, 0.021544535106640447, 0.01719041179349273, 0.016352072151302636, 0.014747076386325825, 0.014607453926486099, 0.013770784739145385, 0.013681774497768327, 0.00637038889585548, 0.00587939608459644, 0.004452300526114682, 0.004446069428440283, 0.004446069428440283, 0.00435890612485206, 0.004248255767193929]"
6,6,"waymo, transdev, av, waymo av, test driver, test, driver, report, passenger vehicle, seating, seating position, position, mode test, mode test driver, driver present","[0.013709534734233489, 0.012252611933974745, 0.011267786534988862, 0.01119170431460361, 0.010224657612082534, 0.010079277638428455, 0.009900643812411137, 0.008934547109946333, 0.0074728285535746305, 0.006451759855167054, 0.006451759855167054, 0.00640300913582592, 0.006270351468116886, 0.006270351468116886, 0.006237528942353856]"
7,7,"waymo, parking, lot, parking lot, waymo av, av, parking lot xxx, lot xxx, arizona, arizona collision, arizona collision involving, lot xxx waymo, mt waymo autonomous, mt, mt waymo","[0.027428167890698792, 0.024359727295003988, 0.022543719093782146, 0.022234661477950845, 0.01677230056

,nmf_topic_0,nmf_topic_1,nmf_topic_2,nmf_topic_3,nmf_topic_4,nmf_topic_5,nmf_topic_6,nmf_topic_7
0,0.035175,0.003412,0.096134,0.000000,0.093597,0.008709,0.023693,0.000000
1,0.000000,0.000000,0.000000,0.204280,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.203874,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.038696,0.001993,0.044977,0.000000,0.000000
4,0.000000,0.019680,0.008555,0.054221,0.013253,0.001174,0.000462,0.000439


### Gensim LDA

In [23]:
# ---------------------------------------------------------------------------
# 3) Tokenize -> gensim LdaModel  (requires `uv pip install gensim`)
# ---------------------------------------------------------------------------
topics_df, doc_topic, doc_index = lda_gensim(
    narratives,
    n_topics=8,
    top_n_words=12,
    no_below=5,
    no_above=0.9,
    passes=10,
)
print('lda_gensim topics_df:', topics_df.shape)
with pd.option_context('display.max_colwidth', None):
    display(topics_df)

# top_weights is a list-in-cell; expand a row to see word + weight pairs.
row = topics_df.iloc[0]
print(list(zip(row['top_words'].split(', '), row['top_weights'])))


ImportError: lda_gensim requires the `gensim` package (uv pip install gensim)

In [ ]:

# ---------------------------------------------------------------------------
# 4) Tokenize + TF-IDF -> gensim Nmf  (requires gensim)
# ---------------------------------------------------------------------------
topics_df, doc_topic, doc_index = nmf_gensim(
    narratives,
    n_topics=8,
    top_n_words=12,
    no_below=5,
    no_above=0.9,
    passes=10,
)
print('nmf_gensim topics_df:', topics_df.shape)
print(topics_df)



## Notes
* sklearn lda: decent topics, matches with the entities at least
* sklearn nmf: seems to get the redacted more concisely but waymo spread out a bunch
* some of the NMF with more ngrams hints more at differe phases of the waymo, may be worht looking ito. ie, lot of parking lot and arizon then more passenger caur and waymo, gm and zoox seem to be isolated